# 14 — Multi-omics: the single-cell findings in a bulk time course, and do organs age together?

The TMS **bulk** atlas (Schaum et al. 2020, `190304_maca_bulk.h5ad`) sequenced 17 organs from ~54 mice at **10 ages** (1, 3, 6, 9, 12, 15, 18, 21, 24, 27 months), ~5 mice per age. Its three sequencing plates are **balanced across ages**, so plate is not tied to age as processing batch is in the single-cell data, and it fills the 3–18 month gap the single-cell data leaves. It is a second data type from the same atlas project: bulk measures every cell of an organ together, so a change can come from composition as well as from inside cells.

1. Do the single-cell findings (MHC class I, complement, collagen loss, p21) hold with 10 ages and plate as a covariate?
2. **When** does p21 rise: early (1–9 months) or late (9–27)?
3. **Do organs age together?** MOFA+ (multi-omics factor analysis) with each organ as a data view and mice as samples, then an "ageotype" test: does a mouse that looks old in one organ look old in the others?

### 1. Setup and load

In [ ]:
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import Ridge
from src import aging as ag

A = ag.paths("atlas")
bulk = ad.read_h5ad(ag.ROOT / "data" / "external" / "190304_maca_bulk.h5ad")
bulk = bulk[bulk.obs["Organ"].astype(str).ne("nan").to_numpy()]
lib = np.asarray(bulk.X.sum(axis=1)).ravel()
bulk = bulk[lib > 1e6].copy()
sc.pp.normalize_total(bulk, target_sum=1e6)
sc.pp.log1p(bulk)                                   # log(1 + counts per million)
obs = bulk.obs[["Organ", "MouseID", "Age", "Sex", "Plate"]].astype({"Organ": str, "MouseID": str, "Sex": str, "Plate": str})
print(bulk.shape, "|", obs["MouseID"].nunique(), "mice |", obs["Organ"].nunique(), "organs")
print("plates per mouse:", obs.groupby("MouseID")["Plate"].nunique().value_counts().to_dict())

### 2. The single-cell findings, re-tested over 10 ages

Per organ and gene: `expression ~ age + plate + sex` (ordinary least squares over that organ's samples, one per mouse), the t-value of age. Plate and sex are covariates, so a plate or sex imbalance can't make the trend. |t| > 2 is a clear trend.

In [ ]:
GENES = ["H2-K1", "H2-D1", "B2m", "C3", "Cfb", "C4b", "Col1a1", "Col3a1", "Cdkn1a", "Cdkn2a"]
X = pd.DataFrame(bulk[:, GENES].X.toarray() if hasattr(bulk.X, "toarray") else np.asarray(bulk[:, GENES].X),
                 columns=GENES, index=bulk.obs_names)
d = pd.concat([obs, X], axis=1)
tvals = {}
for organ, g in d.groupby("Organ"):
    row = {}
    for gene in GENES:
        f = smf.ols(f"Q('{gene}') ~ Age + C(Plate) + C(Sex)", data=g).fit()
        row[gene] = f.tvalues["Age"]
    tvals[organ] = row
tvals = pd.DataFrame(tvals).T
tvals.round(2).to_csv(A.figure("bulk_gene_age_t.csv"))
print("organs with t > 2 / t < -2:")
print(pd.DataFrame({"up": (tvals > 2).sum(), "down": (tvals < -2).sum()}).T)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(tvals, cmap="RdBu_r", center=0, vmin=-8, vmax=8, annot=True, fmt=".1f", annot_kws={"size": 6}, ax=ax,
            cbar_kws={"label": "t of age (plate, sex adjusted)"})
fig.savefig(A.figure("bulk_gene_age_t.png"), dpi=200, bbox_inches="tight")

### 3. When does p21 rise?

Single-cell TMS showed p21 higher at 18+ months than at 1–3 months; Kimmel et al. 2019 saw no change from 7 to 21 months. Split the bulk ages: **early** 1–9 months and **late** 9–27 months, Spearman ρ of *Cdkn1a* with age within each, per organ. If the rise is early, ρ is positive early and flat late. *Cdkn2a* (p16), barely seen per cell, for comparison.

In [ ]:
def window(gene, lo, hi):
    out = {}
    for organ, g in d[(d["Age"] >= lo) & (d["Age"] <= hi)].groupby("Organ"):
        r, p = spearmanr(g["Age"], g[gene])
        out[organ] = r
    return pd.Series(out)

timing = pd.DataFrame({"p21_early_1_9": window("Cdkn1a", 1, 9), "p21_late_9_27": window("Cdkn1a", 9, 27),
                       "p16_early_1_9": window("Cdkn2a", 1, 9), "p16_late_9_27": window("Cdkn2a", 9, 27)})
timing.round(2).to_csv(A.figure("bulk_p21_p16_timing.csv"))
print("median across organs:", timing.median().round(2).to_dict())
print("organs with rho > 0.3:", (timing > 0.3).sum().to_dict())
timing.round(2)

In [ ]:
means = d.groupby(["Organ", "Age"])[["Cdkn1a", "Cdkn2a"]].mean().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, gene in zip(axes, ["Cdkn1a", "Cdkn2a"]):
    for organ, g in means.groupby("Organ"):
        ax.plot(g["Age"], g[gene] - g[gene].iloc[0], lw=1, alpha=0.7, label=organ)
    ax.axvspan(3, 7.5, color="gray", alpha=0.15)
    ax.set_xlabel("age (months)"); ax.set_ylabel(f"{gene}: log CPM minus 1-month value")
    ax.set_title(f"{gene} ({'p21' if gene == 'Cdkn1a' else 'p16'}); grey = window Kimmel's young mice sit after")
axes[1].legend(fontsize=6, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.savefig(A.figure("bulk_p21_p16_trajectory.png"), dpi=200, bbox_inches="tight")

### 4. MOFA+: factors shared across organs

Each organ is one data view (its 1,000 most variable genes, centered); each mouse is one sample, so an organ a mouse didn't give is simply missing. MOFA+ finds a small number of latent factors, each a pattern over mice, and reports how much of each organ's variation each factor explains. A factor that tracks **age** and explains variation in **many organs** is a body-wide aging axis. Factors are then checked against age, sex and plate.

In [ ]:
from mofapy2.run.entry_point import entry_point
import h5py, io, contextlib

rows = []
for organ, idx in obs.groupby("Organ").indices.items():
    sub = bulk[idx]
    Xo = sub.X.toarray() if hasattr(sub.X, "toarray") else np.asarray(sub.X)
    top = np.argsort(Xo.var(axis=0))[::-1][:1000]
    M = pd.DataFrame(Xo[:, top] - Xo[:, top].mean(axis=0), index=obs.iloc[idx]["MouseID"].to_numpy(),
                     columns=organ + ":" + sub.var_names[top])
    M = M.groupby(level=0).mean()                         # one sample per mouse per organ
    long = M.stack().rename("value").reset_index()
    long.columns = ["sample", "feature", "value"]
    rows.append(long.assign(view=organ, group="all"))
long = pd.concat(rows, ignore_index=True)

with contextlib.redirect_stdout(io.StringIO()):     # MOFA+ prints every iteration
    ent = entry_point()
    ent.set_data_options(scale_views=True)
    ent.set_data_df(long)
    ent.set_model_options(factors=10)
    ent.set_train_options(iter=1000, convergence_mode="fast", seed=0, verbose=False)
    ent.build()
    ent.run()
out = str(A.figure("mofa_bulk.hdf5"))
with contextlib.redirect_stdout(io.StringIO()):
    ent.save(out)
with h5py.File(out) as f:
    Z = pd.DataFrame(f["expectations/Z/all"][:].T, index=[s.decode() for s in f["samples/all"][:]],
                     columns=[f"F{i + 1}" for i in range(f["expectations/Z/all"].shape[0])])
    views = [v.decode() for v in f["views/views"][:]]
    r2 = pd.DataFrame(f["variance_explained/r2_per_factor/all"][:], index=views, columns=Z.columns)

In [ ]:
mice = obs.drop_duplicates("MouseID").set_index("MouseID").loc[Z.index]
assoc = pd.DataFrame({
    "rho_age": [spearmanr(Z[c], mice["Age"])[0] for c in Z],
    "p_age": [spearmanr(Z[c], mice["Age"])[1] for c in Z],
    "sex_diff": [abs(Z.loc[mice["Sex"] == "m", c].mean() - Z.loc[mice["Sex"] == "f", c].mean()) / Z[c].std() for c in Z],
    "organs_r2_above_2pct": (r2 > 2).sum(),
    "mean_r2_pct": r2.mean().round(2)}, index=Z.columns)
assoc.round(3).to_csv(A.figure("mofa_factor_association.csv"))
age_f = assoc["rho_age"].abs().idxmax()
print("strongest age factor:", age_f)
display(assoc.round(3))
print(f"\nvariance explained by {age_f}, per organ (%):")
r2[age_f].sort_values(ascending=False).round(2)

### 5. Ageotypes: does a mouse that looks old in one organ look old in the others?

Per organ, a ridge regression on the 2,000 most variable genes predicts each mouse's age, leave-one-mouse-out. The **residual** (predicted minus true age) says how old that organ looks for its age. If aging is coordinated across the body, residuals of different organs correlate across mice. Two artifacts could fake that: the ridge pulls predictions toward the mean age (residuals fall with true age in every organ alike), so residuals are first regressed on true age; organs of one mouse could share a plate, and one sex could look older throughout; so residuals are also adjusted for plate and sex (`resid_noplate`), and the test is repeated within each sex.

In [ ]:
from sklearn.model_selection import LeaveOneOut
resid = {}
for organ, idx in obs.groupby("Organ").indices.items():
    sub = bulk[idx]
    Xo = sub.X.toarray() if hasattr(sub.X, "toarray") else np.asarray(sub.X)
    top = np.argsort(Xo.var(axis=0))[::-1][:2000]
    Xo = Xo[:, top]
    y = sub.obs["Age"].to_numpy().astype(float)
    m_ids = sub.obs["MouseID"].astype(str).to_numpy()
    pred = np.zeros(len(y))
    for tr, te in LeaveOneOut().split(Xo):
        pred[te] = Ridge(alpha=100).fit(Xo[tr], y[tr]).predict(Xo[te])
    r = pred - y
    df = pd.DataFrame({"r": pred - y, "age": y, "sex": sub.obs["Sex"].astype(str).to_numpy(),
                       "plate": sub.obs["Plate"].astype(str).to_numpy()}, index=m_ids)
    resid_age = smf.ols("r ~ age", data=df).fit().resid                           # regression to the mean
    resid_all = smf.ols("r ~ age + C(plate) + C(sex)", data=df).fit().resid       # also plate and sex
    resid[organ] = pd.DataFrame({"resid": resid_age, "resid_noplate": resid_all, "sex": df["sex"],
                                 "mae": np.abs(pred - y)}, index=m_ids)
mae = pd.Series({o: v["mae"].mean() for o, v in resid.items()}).sort_values()
print("clock error per organ (months, leave-one-mouse-out):"); print(mae.round(1).to_dict())

def pairs(col):
    R = pd.DataFrame({o: v[col] for o, v in resid.items()})
    C = R.corr(min_periods=20)
    return R, C.where(np.triu(np.ones(C.shape, bool), 1)).stack().dropna()   # pandas 3 keeps NaN in stack

R, c_all = pairs("resid")
_, c_np = pairs("resid_noplate")
rng = np.random.default_rng(0)
null = []
for _ in range(200):                                       # shuffle mice independently per organ
    Rs = R.apply(lambda s: pd.Series(rng.permutation(s.dropna().to_numpy()), index=s.dropna().index))
    Cs = Rs.corr(min_periods=20)
    null.append(Cs.where(np.triu(np.ones(Cs.shape, bool), 1)).stack().dropna().median())
print(f"median organ-pair residual correlation: {c_all.median():.3f} (plate removed: {c_np.median():.3f}); "
      f"shuffled-mice null median {np.median(null):.3f}, 95th percentile {np.quantile(null, 0.95):.3f}; "
      f"pairs > 0: {(c_np > 0).mean():.2f}")
pd.DataFrame({"rho": c_all, "rho_no_plate": c_np}).sort_values("rho_no_plate", ascending=False).round(2) \
  .to_csv(A.figure("ageotype_pairs.csv"))

In [ ]:
for sx in ["m", "f"]:
    Rs = pd.DataFrame({o: v.loc[v["sex"] == sx, "resid_noplate"] for o, v in resid.items()})
    Cs = Rs.corr(min_periods=12)
    cs = Cs.where(np.triu(np.ones(Cs.shape, bool), 1)).stack().dropna()
    print(f"{'males' if sx == 'm' else 'females'} only: median {cs.median():.3f}, pairs > 0: {(cs > 0).mean():.2f}, pairs: {len(cs)}")

In [ ]:
C = pd.DataFrame({o: v["resid_noplate"] for o, v in resid.items()}).corr(min_periods=20)
g = sns.clustermap(C, cmap="RdBu_r", center=0, vmin=-0.6, vmax=0.6, figsize=(8, 8))
g.ax_heatmap.set_title("organ age residuals, correlated across mice")
g.savefig(A.figure("ageotype_organ_correlation.png"), dpi=200, bbox_inches="tight")

### Findings

922 bulk samples, 17 organs, 56 mice, 10 ages (1–27 months); plates balanced across ages; plate and sex as covariates.

- **Matrix loss is the most universal aging change.** *Col1a1* and *Col3a1* fall with age in 13 and 14 of 17 organs (t down to −10 in limb muscle), and rise in none. This replicates the single-cell stromal collagen decline with ten ages and no plate–age confounding. Bulk alone cannot say whether fibroblasts make less collagen or there are fewer fibroblasts; the single-cell lung stroma (05b, 09) shows the per-cell fall.
- **Immune recognition and complement rise in most organs.** MHC class I (*H2-K1*, *H2-D1*, *B2m*) up in 6–7 organs (brain, kidney, heart, lung, skin, blood) and down in 2 (subcutaneous fat, small intestine); complement *C4b* up in 9 (brain t = 18, kidney 11), *C3* in 5. Bulk can't say whether this is more immune cells or more expression per cell; single-cell (09) says MHC class I rises inside cells.
- **p21 does not rise with age in bulk**, early or late: up in 1 organ and down in 5 (brown fat t −5.6), median ρ −0.08 over 1–9 months and −0.13 over 9–27. With Kimmel et al. 2019 also flat, the p21 rise of the single-cell atlas **does not replicate**. Bulk is a whole organ, so a rise confined to fibroblasts would be diluted (lung bulk p21 t = −1.9); the strongest counter-evidence is therefore Kimmel's sorted cell types, which are small (3 vs 3–4 mice). Read as: not replicated, cause unknown — not as proven artifact. p16 (*Cdkn2a*) rises weakly, in 3 organs.
- **A body-wide aging axis.** MOFA+ over 17 organs finds one factor (F2) that tracks age at ρ = 0.95 and explains variation in 15 of 17 organs, most in fat depots (mesenteric 18%, gonadal 16%), limb muscle (14%) and lung (9%); least in pancreas, marrow, brain, spleen (< 3%). Another factor (F1) is sex.
- **Organs of one mouse look old together.** Per-organ clocks predict age to 2.1–4.7 months (kidney best, pancreas worst). After removing regression to the mean, plate and sex, the residuals of 93% of organ pairs correlate positively across mice (median 0.23; shuffled-mice null 95th percentile 0.03), within males (0.22) and females (0.33) alike; strongest pairs brown fat–liver (0.62), liver–skin (0.56), kidney–pancreas (0.54). This is the "ageotype" pattern: some mice are biologically older than others, across organs. **Caveat:** all organs of a mouse were dissected together, so a per-mouse handling effect would look the same; this data cannot separate the two.